# 第04章　矩阵的解剖：D组整合实验

本 Notebook 对统稿中的四节核心结论做可重复的数值检查。运行环境为 Python 3.12，
核心依赖为 NumPy、SciPy、Matplotlib 与 SymPy。执行结果由重新启动的内核顺序运行后保存。

基线：`main@c1f4d2b45178ae28bdbacfaedfdfdf8cd1a98839`。主稿和补充稿的选择见同章 `整合-正文.md` 与 D组报告。


## 4.1 直接法：LU 与 Cholesky

比较一般矩阵的 LU 分解与对称正定矩阵的 Cholesky 分解。

In [1]:
import json
import numpy as np
import scipy.linalg as la

A = np.array([[4., 2., 1.], [3., 5., 2.], [1., 1., 3.]])
P, L, U = la.lu(A)
b = np.array([7., 10., 5.])
x_lu = la.solve(A, b)
print("LU 残差:", np.linalg.norm(A@x_lu-b, ord=np.inf))
S = np.array([[4., 2.], [2., 5.]])
C = la.cholesky(S, lower=True)
print("Cholesky 重构误差:", np.linalg.norm(C@C.T-S, ord=np.inf))
assert np.linalg.norm(P@A-L@U, ord=np.inf) < 1e-12
assert np.linalg.norm(C@C.T-S, ord=np.inf) < 1e-12

LU 残差: 0.0
Cholesky 重构误差: 0.0


## 4.2 共轭梯度法

对对称正定矩阵运行 CG，并检查残差持续下降。

In [2]:
def conjugate_gradient(A, b, max_iter=50, tol=1e-12):
    x = np.zeros_like(b)
    r = b-A@x
    p = r.copy()
    history = [np.linalg.norm(r)]
    for _ in range(max_iter):
        Ap = A@p
        alpha = np.dot(r, r)/np.dot(p, Ap)
        x = x+alpha*p
        r_new = r-alpha*Ap
        history.append(np.linalg.norm(r_new))
        if history[-1] < tol:
            break
        beta = np.dot(r_new, r_new)/np.dot(r, r)
        p = r_new+beta*p
        r = r_new
    return x, np.asarray(history)

M = np.array([[4., 1.], [1., 3.]])
rhs = np.array([1., 2.])
cg_solution, residual_history = conjugate_gradient(M, rhs)
print("CG 解:", cg_solution)
print("CG 残差:", residual_history)
assert np.linalg.norm(cg_solution-np.linalg.solve(M, rhs)) < 1e-10

CG 解: [0.09090909 0.63636364]
CG 残差: [2.23606798 0.55901699 0.        ]


## 4.3 SVD、条件数与低秩近似

检查奇异值、条件数和截断 SVD 的最优低秩误差。

In [3]:
matrix = np.array([[3., 1.], [1., 2.]])
U, singular_values, Vt = np.linalg.svd(matrix)
rank_one = singular_values[0]*np.outer(U[:, 0], Vt[0, :])
rank_one_error = np.linalg.norm(matrix-rank_one, ord=2)
print("奇异值:", singular_values)
print("条件数:", np.linalg.cond(matrix))
print("最佳秩1误差:", rank_one_error)
assert np.isclose(rank_one_error, singular_values[1])

奇异值: [3.61803399 1.38196601]
条件数: 2.618033988749896
最佳秩1误差: 1.3819660112501053


## 4.4 可微求解器、PCA 与 LoRA

用有限差分核对伴随梯度，并检查 PCA 中心化与 LoRA 参数量结论。

In [4]:
Asolve = np.array([[2., 1.], [1., 3.]])
rhs = np.array([1., 2.])
x = np.linalg.solve(Asolve, rhs)
z = np.linalg.solve(Asolve.T, x)
gradient_b = z
gradient_A = -np.outer(z, x)

def loss(value):
    solution = np.linalg.solve(Asolve, value)
    return 0.5*np.dot(solution, solution)

epsilon = 1e-6
finite_difference = np.zeros(2)
for index in range(2):
    plus = rhs.copy(); plus[index] += epsilon
    minus = rhs.copy(); minus[index] -= epsilon
    finite_difference[index] = (loss(plus)-loss(minus))/(2*epsilon)
print("伴随梯度:", gradient_b)
print("有限差分:", finite_difference)
assert np.allclose(gradient_b, finite_difference, atol=1e-8)

mean = np.array([10., -5.])
scatter_direction = np.array([1., 1.])/np.sqrt(2)
samples = mean + np.linspace(-2, 2, 101)[:, None]*scatter_direction
raw_cov = samples.T@samples/len(samples)
centered = samples-samples.mean(axis=0)
centered_cov = centered.T@centered/len(centered)
raw_direction = np.linalg.eigh(raw_cov)[1][:, -1]
centered_direction = np.linalg.eigh(centered_cov)[1][:, -1]
print("未中心化主轴:", raw_direction)
print("中心化主轴:", centered_direction)
assert abs(np.dot(raw_direction, mean/np.linalg.norm(mean))) > 0.99
assert abs(np.dot(centered_direction, scatter_direction)) > 0.99

k, d = 256, 128
full = k*d
lora = {r: r*(k+d) for r in [4, 8, 16]}
print("LoRA 参数量:", lora)
assert lora[8] == 3072

伴随梯度: [0.  0.2]
有限差分: [0.  0.2]
未中心化主轴: [-0.89589484  0.44426618]
中心化主轴: [0.70710678 0.70710678]
LoRA 参数量: {4: 1536, 8: 3072, 16: 6144}


## 执行记录

以下单元把关键结果写入同章 `数据/整合-实验摘要.json`，供网页、报告和后续复核共用。

In [5]:
summary = {
    "lu_residual": float(np.linalg.norm(A@x_lu-b, ord=np.inf)),
    "cholesky_error": float(np.linalg.norm(C@C.T-S, ord=np.inf)),
    "cg_iterations": int(len(residual_history)-1),
    "singular_values": singular_values.tolist(),
    "condition_number": float(np.linalg.cond(matrix)),
    "rank_one_error": float(rank_one_error),
    "adjoint_gradient": gradient_b.tolist(),
    "finite_difference": finite_difference.tolist(),
    "lora_parameters": lora,
}
data_dir = __import__("pathlib").Path("数据")
data_dir.mkdir(exist_ok=True)
(data_dir/"整合-实验摘要.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)+"\n",
    encoding="utf-8",
)
print("整合实验摘要已保存到 数据/整合-实验摘要.json")

整合实验摘要已保存到 数据/整合-实验摘要.json


## 结论与边界

本 Notebook 验证的是统稿中的核心数值结论和可复现性；它不替代定理证明、A 组人工判断
或 B/C 的独立复核。图表与结果均来自本次重启内核后的顺序执行，没有沿用源 Notebook 输出。